# Task 1: Booking Abandonment & Cancellation Risk Prediction
Trains a scikit-learn classification pipeline on data/hotel_bookings_cleaned.csv mapped to the FastAPI input schema (Booking_Channel, Guest_Segment, Room_Type, Stay_Length_Days, Nightly_Rate_NGN, Lead_Time_Days) and exports rtifacts/abandonment_model.joblib.

In [1]:
import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df = pd.read_csv('../data/hotel_bookings_cleaned.csv')
print('Dataset shape:', df.shape)

In [2]:
channel_map = {
    'Direct': 'Website', 'Online TA': 'OTA', 'Offline TA/TO': 'Travel Agent',
    'Corporate': 'Corporate', 'Groups': 'Group', 'Complementary': 'Website',
    'Aviation': 'Corporate', 'Undefined': 'Website'
}
room_map = {
    'A': 'Standard', 'B': 'Standard', 'C': 'Deluxe', 'D': 'Deluxe',
    'E': 'Suite', 'F': 'Suite', 'G': 'Executive', 'H': 'Executive',
    'L': 'Standard', 'P': 'Standard'
}

def derive_segment(row):
    if row['market_segment'] == 'Corporate': return 'Business'
    if row['market_segment'] == 'Groups': return 'Group'
    if (row['children'] + row['babies']) > 0: return 'Family'
    if row['adults'] == 1: return 'Solo'
    return 'Couple'

model_df = pd.DataFrame({
    'Booking_Channel': df['market_segment'].map(channel_map).fillna('Website'),
    'Guest_Segment': df.apply(derive_segment, axis=1),
    'Room_Type': df['reserved_room_type'].map(room_map).fillna('Standard'),
    'Stay_Length_Days': df['total_nights'].clip(lower=1),
    'Nightly_Rate_NGN': (df['adr'].clip(lower=10.0) * 1500.0).round(2),
    'Lead_Time_Days': df['lead_time'].clip(lower=0),
    'is_abandoned': df['is_canceled'].astype(int),
})
model_df.head()

In [3]:
feature_cols = [
    'Booking_Channel', 'Guest_Segment', 'Room_Type',
    'Stay_Length_Days', 'Nightly_Rate_NGN', 'Lead_Time_Days'
]
X = model_df[feature_cols]
y = model_df['is_abandoned']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), ['Booking_Channel', 'Guest_Segment', 'Room_Type']),
    ('num', StandardScaler(), ['Stay_Length_Days', 'Nightly_Rate_NGN', 'Lead_Time_Days']),
])

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, max_depth=16, min_samples_leaf=5, random_state=42, n_jobs=-1)),
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]
print(f'Validation Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print(f'Validation ROC-AUC:  {roc_auc_score(y_test, y_prob):.4f}')
print(classification_report(y_test, y_pred))

Validation Accuracy: 0.7824
Validation ROC-AUC:  0.8446
              precision    recall  f1-score   support

           0       0.76      0.95      0.85     15033
           1       0.84      0.51      0.63      8845

    accuracy                           0.78     23878
   macro avg       0.80      0.73      0.74     23878
weighted avg       0.79      0.78      0.77     23878


In [4]:
joblib.dump(pipeline, '../artifacts/abandonment_model.joblib')
print('Saved ../artifacts/abandonment_model.joblib successfully!')

Saved ../artifacts/abandonment_model.joblib successfully!
